In [33]:
import torch
from pubmed_corpus import pubmed_corpus
import gradio as gr
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from sentence_transformers import SentenceTransformer, util

/Users/efeemirhandogan/.pyenv/versions/3.11.5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE

device(type='cpu')

In [4]:
bert_model = SentenceTransformer('bert-base-nli-mean-tokens')
bio_bert_model = SentenceTransformer('pritamdeka/S-BioBERT-snli-multinli-stsb')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6288.55it/s]
BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48728.27it/s]
BertModel LOAD REPORT from: pritamdeka/S-BioBERT-snli-multinli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
corpus = [
    "Aspirin is used to reduce fever and relieve mild to moderate pain.",
    "Apoptosis is a form of programmed cell death that occurs in multicellular organisms.",
    "The patient was diagnosed with severe acute respiratory syndrome (SARS).",
    "Machine learning models are heavily used in natural language processing." # Unrelated Document
]

In [6]:
query = "What is the treatment for elevated body temperature and physical discomfort?"

In [7]:
# Encoding with BERT 
corpus_embeddings_bert = bert_model.encode(corpus, convert_to_tensor=True)
query_embedding_bert = bert_model.encode(query, convert_to_tensor=True)

# Encoding with BioBERT
corpus_embeddings_bio_bert = bio_bert_model.encode(corpus, convert_to_tensor=True)
query_embedding_bio_bert = bio_bert_model.encode(query, convert_to_tensor=True)

In [ ]:
def calculate_results(original_corpus,corpus_embedding, query_embedding):
    cos_sim = util.cos_sim(query_embedding, corpus_embedding)[0]
    top_results = torch.topk(cos_sim, k=2)

    for score, idx in zip(top_results[0], top_results[1]):
        print(f"Score: {score:.4f} \t Document ID: {idx} \t Document: {original_corpus[idx]}")

In [8]:
# BERT Results
print("--- Standart BERT Results ---")
cos_scores_bert = util.cos_sim(query_embedding_bert, corpus_embeddings_bert)[0]
top_results_bert = torch.topk(cos_scores_bert, k=2)

for score, idx in zip(top_results_bert[0], top_results_bert[1]):
    print(f"Score: {score:.4f} \t Document: {corpus[idx]}")

# BioBERT Results
print("\n--- BioBERT Results ---")
cos_scores_biobert = util.cos_sim(query_embedding_bio_bert, corpus_embeddings_bio_bert)[0]
top_results_biobert = torch.topk(cos_scores_biobert, k=2)

for score, idx in zip(top_results_biobert[0], top_results_biobert[1]):
    print(f"Score: {score:.4f} \t Document: {corpus[idx]}")

--- Standart BERT Results ---
Score: 0.5443 	 Document: The patient was diagnosed with severe acute respiratory syndrome (SARS).
Score: 0.5032 	 Document: Aspirin is used to reduce fever and relieve mild to moderate pain.

--- BioBERT Results ---
Score: 0.3844 	 Document: Aspirin is used to reduce fever and relieve mild to moderate pain.
Score: 0.2070 	 Document: The patient was diagnosed with severe acute respiratory syndrome (SARS).


In [12]:
pubmed_query = "Treatments for progressive movement disorders characterized by severe shaking and slowness of movement."

In [23]:
pubmed_query = "A condition causing extreme continuous thirst and excessive production of highly dilute urine due to hormone issues."

In [26]:
pubmed_query = "Using hallucinogenic mushroom extracts as a therapeutic approach to treat severe trauma and mental health disorders in survivors."

In [29]:
pubmed_query = "Acute kidney failure and renal dysfunction requiring hemodialysis after eating poisonous fungi."

In [30]:
# Encoding pubmed corpus with BERT
pubmed_corpus_embed_bert = bert_model.encode(pubmed_corpus, convert_to_tensor=True)
pubmed_query_embed_bert = bert_model.encode(pubmed_query, convert_to_tensor=True)

# Encoding pubmed corpus with BioBERT
pubmed_corpus_embed_biobert = bio_bert_model.encode(pubmed_corpus, convert_to_tensor=True)
pubmed_query_embed_biobert = bio_bert_model.encode(pubmed_query, convert_to_tensor=True)

In [31]:
print("BERT results for PubMed Corpus:\n")
calculate_results(pubmed_corpus,pubmed_corpus_embed_bert, pubmed_query_embed_bert)
print("\nBioBERT results for PubMed Corpus:\n")
calculate_results(pubmed_corpus,pubmed_corpus_embed_biobert, pubmed_query_embed_biobert)

BERT results for PubMed Corpus:

Score: 0.7265 	 Document ID: 2 	 Document: Central diabetes insipidus (CDI) is a clinical syndrome which results from loss or impaired function of vasopressinergic neurons in the hypothalamus/posterior pituitary, resulting in impaired synthesis and/or secretion of arginine vasopressin (AVP). AVP deficiency leads to the inability to concentrate urine and excessive renal water losses, resulting in a clinical syndrome of hypotonic polyuria with compensatory thirst. CDI is caused by diverse etiologies, although it typically develops due to neoplastic, traumatic, or autoimmune destruction of AVP-synthesizing/secreting neurons. This review focuses on the diagnosis and management of CDI, providing insights into the physiological disturbances underpinning the syndrome. Recent developments in diagnostic techniques, particularly the development of the copeptin assay, have improved accuracy and acceptability of the diagnostic approach to the hypotonic polyuria syn

In [49]:
def get_results(query, model, corpus_embedding, k=2):
    # Query to embedding
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Compute cosine similarity
    cos_scores = util.cos_sim(query_embedding, corpus_embedding)[0]

    # Get top k results
    top_results = torch.topk(cos_scores, k=k)

    # results to markdown
    result_markdown = ""
    for rank, (score, idx) in enumerate(zip(top_results[0], top_results[1]), 1):
        idx_val = idx.item()
        score_val = score.item()
        
        result_markdown += f"### Rank {rank} | Documnet ID: {idx_val} | Score: {score_val:.4f}\n"
        result_markdown += f"> {pubmed_corpus[idx_val]}\n\n---\n\n"
        
    return result_markdown

In [38]:
def compare_models(query):
    bert_results = get_results(query, bert_model, pubmed_corpus_embed_bert)
    biobert_results = get_results(query, bio_bert_model, pubmed_corpus_embed_biobert)
    
    return bert_results, biobert_results

In [50]:
# Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>🧬 BERT vs. BioBERT: Biomedical Retrieval Compare</h1>")
    gr.Markdown("<p style='text-align: center;'>Enter your search query and compare the results delivered by general-purpose BERT and medically trained BioBERT side-by-side.</p>")
    
    # Upper part: Query input and search button
    with gr.Row():
        with gr.Column(scale=4):
            query_input = gr.Textbox(
                label="Query", 
                placeholder="Ex: Treatments for progressive movement disorders characterized by severe shaking...",
                lines=2
            )
        with gr.Column(scale=1):
            search_btn = gr.Button("Get Results", variant="primary")
            
    # Example queries
    gr.Examples(
        examples=[
            "Treatments for progressive movement disorders characterized by severe shaking and slowness of movement.",
            "A condition causing extreme continuous thirst and excessive production of highly dilute urine due to hormone issues.",
            "Acute kidney failure and renal dysfunction requiring hemodialysis after eating poisonous fungi."
        ],
        inputs=query_input
    )
            
    # Bottom part: Results display
    with gr.Row():
        # Left Column (Standart BERT)
        with gr.Column():
            gr.Markdown("<h2 style='color: #d9534f;'>Standart BERT Results</h2>")
            bert_output = gr.Markdown("Results will be displayed here...")
            
        # Right Column (BioBERT)
        with gr.Column():
            gr.Markdown("<h2 style='color: #5cb85c;'>BioBERT Results</h2>")
            biobert_output = gr.Markdown("Results will be displayed here...")

    # Button click event
    search_btn.click(
        fn=compare_models, 
        inputs=query_input, 
        outputs=[bert_output, biobert_output]
    )

/var/folders/_d/9nvwwyt97xs156nmz906c5pw0000gn/T/ipykernel_44651/1093956390.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [46]:
demo.close()

Closing server running on port: 7864


In [48]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
